# Keep the Property vs. Sell and Invest

Two scenarios compared over a long horizon:

- **Keep**: Property held as a leveraged real-estate investment. The mortgage position is modeled via `LeveragedAsset`. Net rental income modeled as a monthly cashflow contribution. Property costs (tax, HOA, maintenance) as a monthly outflow.
- **Sell**: Property sold, proceeds invested into a diversified portfolio (equities + bonds). No property-specific cashflows.

The liquid-asset portion of both portfolios is identical, so the comparison isolates the property decision.

**How to use**: update the parameters in the cell below, then run all cells.

In [1]:
from datetime import date
import polars as pl
import waypoint as wp

## Parameters

Update these to match your situation.

In [2]:
# ── Property economics ────────────────────────────────────────────────────
PROPERTY_VALUE        = 500_000.0   # current market value
MORTGAGE_BALANCE      = 300_000.0   # outstanding mortgage principal
PROPERTY_EQUITY       = PROPERTY_VALUE - MORTGAGE_BALANCE

MORTGAGE_RATE         = 0.065       # annual financing cost (mortgage interest rate)
LEVERAGE_RATIO        = PROPERTY_VALUE / PROPERTY_EQUITY

MONTHLY_RENT_NET      =  2_000.0    # net rental income after vacancy / management fees

# Monthly property costs — enter as a positive number; it will be negated below
MONTHLY_PROPERTY_TAX  =    400.0    # property tax / 12
MONTHLY_HOA           =    200.0    # HOA / condo fee (0 if detached)
MONTHLY_MAINTENANCE   =    150.0    # repairs & maintenance reserve
MONTHLY_PROPERTY_COST = -1 * (MONTHLY_PROPERTY_TAX + MONTHLY_HOA + MONTHLY_MAINTENANCE)

# ── Proceeds when selling ─────────────────────────────────────────────────
CLOSING_COST_PCT  = 0.05   # ~5% of sale price (agent fees, transfer tax, etc.)
SELL_PROCEEDS     = PROPERTY_VALUE * (1 - CLOSING_COST_PCT) - MORTGAGE_BALANCE

# ── Shared portfolio parameters ───────────────────────────────────────────
LIQUID_WEALTH    = 200_000.0   # other investable assets (identical in both scenarios)
ANNUAL_SAVINGS   =  20_000.0   # recurring annual contributions (same in both scenarios)
INFLATION_RATE   = 0.03        # annual

# ── Historical data window ────────────────────────────────────────────────
HIST_START    = "2010-01-01"
HIST_END      = "2025-12-31"

# ── Simulation parameters ─────────────────────────────────────────────────
HORIZON_YEARS = 20
N_SIMS        = 2_000
TODAY         = date.today()
# ─────────────────────────────────────────────────────────────────────────

print(f"Property equity: ${PROPERTY_EQUITY:,.0f}")
print(f"Leverage ratio:  {LEVERAGE_RATIO:.2f}×")
print(f"Sell proceeds:   ${SELL_PROCEEDS:,.0f}")

Property equity: $200,000
Leverage ratio:  2.50×
Sell proceeds:   $175,000


## Fetch historical data

Replace `wp.catalog.real_estate.BOSTON_HPI` with a region-appropriate HPI from `wp.catalog.real_estate`.

In [3]:
# Real estate: regional house price index (quarterly, FHFA/FRED)
# Swap this for the index that matches your property's market
local_hpi = wp.fetch(wp.catalog.real_estate.BOSTON_HPI, start=HIST_START, end=HIST_END)

# Equity components (shared across both scenarios)
us_total_mkt  = wp.fetch(wp.catalog.equities.US_TOTAL_MARKET,     start=HIST_START, end=HIST_END)
us_lg_growth  = wp.fetch(wp.catalog.equities.US_LARGE_CAP_GROWTH, start=HIST_START, end=HIST_END)
nasdaq        = wp.fetch(wp.catalog.equities.NASDAQ_100,           start=HIST_START, end=HIST_END)
russell_1000  = wp.fetch(wp.catalog.equities.RUSSELL_1000,         start=HIST_START, end=HIST_END)
europe        = wp.fetch(wp.catalog.equities.EUROPE_DEVELOPED,     start=HIST_START, end=HIST_END)
us_large_cap  = wp.fetch(wp.catalog.equities.US_LARGE_CAP,         start=HIST_START, end=HIST_END)

$^SPX: possibly delisted; no price data found  (1d 2010-01-01 -> 2010-01-04)


## Build the leveraged property asset

`LeveragedAsset` applies the constant-leverage formula each period:
```
r_levered = leverage_ratio × r_asset − (leverage_ratio − 1) × (mortgage_rate / periods_per_year)
```

In [4]:
leveraged_property = wp.LeveragedAsset(
    asset=local_hpi,
    leverage_ratio=LEVERAGE_RATIO,
    financing_cost=MORTGAGE_RATE,
    name="Property (Leveraged)",
)

print(f"Unlevered mean quarterly return: {local_hpi.returns['returns'].mean():.4f}")
print(f"Levered mean quarterly return:   {leveraged_property.returns['returns'].mean():.4f}")

Unlevered mean quarterly return: 0.0125
Levered mean quarterly return:   0.0069


## Scenario A — Keep the property

Portfolio:
- Property (leveraged) — weighted by equity share of total wealth
- Liquid equity and bond holdings (same assets as Scenario B)

Cashflows:
- Monthly net rent income (inflow)
- Monthly property costs — tax, HOA, maintenance (outflow)
- Annual savings contributions (same as Scenario B)

In [5]:
# Total wealth in Scenario A = equity + liquid holdings
TOTAL_WEALTH_KEEP   = PROPERTY_EQUITY + LIQUID_WEALTH
LIQUID_WEIGHT_KEEP  = LIQUID_WEALTH / TOTAL_WEALTH_KEEP
PROPERTY_WEIGHT     = PROPERTY_EQUITY / TOTAL_WEALTH_KEEP

# Liquid slots — contributions and rent income route here (not to the illiquid property)
LIQUID_SLOTS = (
    "US Total Market",
    "US Large Cap Growth",
    "NASDAQ 100",
    "Russell 1000",
    "Europe Developed",
    "US Large Cap",
)

# Liquid sub-weights (must sum to 1.0 within liquid portion)
LIQUID_SUB_WEIGHTS = {
    "US Total Market":     0.40,
    "US Large Cap Growth": 0.38,
    "NASDAQ 100":          0.07,
    "Russell 1000":        0.07,
    "Europe Developed":    0.04,
    "US Large Cap":        0.04,
}

keep_weights = {name: w * LIQUID_WEIGHT_KEEP for name, w in LIQUID_SUB_WEIGHTS.items()}
keep_weights["Property"] = PROPERTY_WEIGHT

keep_portfolio = wp.Portfolio(
    slots={
        "Property":            leveraged_property,
        "US Total Market":     us_total_mkt,
        "US Large Cap Growth": us_lg_growth,
        "NASDAQ 100":          nasdaq,
        "Russell 1000":        russell_1000,
        "Europe Developed":    europe,
        "US Large Cap":        us_large_cap,
    },
    weights=keep_weights,
    name="Keep Property",
    normalize_weights=False,
)

keep_cashflows = [
    # Net rental income (inflow, nominal — rent and costs roughly track each other)
    wp.cashflows.PeriodicCashflow(
        amount=MONTHLY_RENT_NET,
        frequency="monthly",
        mode="dollar",
        real=True,
        slots=LIQUID_SLOTS,
    ),
    # Property costs: tax + HOA + maintenance (outflow)
    wp.cashflows.PeriodicCashflow(
        amount=MONTHLY_PROPERTY_COST,
        frequency="monthly",
        mode="dollar",
        real=True,
        slots=LIQUID_SLOTS,
    ),
    # Annual savings contributions
    wp.cashflows.PeriodicCashflow(
        amount=ANNUAL_SAVINGS,
        frequency="annual",
        mode="dollar",
        real=True,
        slots=LIQUID_SLOTS,
    ),
]

keep_sim = wp.analytics.WealthSimulation(
    method=wp.sim.MonteCarlo(seed=42),
    cashflows=keep_cashflows,
    horizon_years=HORIZON_YEARS,
    initial_wealth=TOTAL_WEALTH_KEEP,
    n_simulations=N_SIMS,
    inflation_rate=INFLATION_RATE,
)

keep_result = keep_sim.compute(
    keep_portfolio,
    start=HIST_START,
    end=HIST_END,
    frequency="quarterly",   # match the quarterly HPI data
    start_date=TODAY,
    real=True,
)

print(f"Keep — initial wealth: ${TOTAL_WEALTH_KEEP:,.0f}")
print(f"  Property weight:  {PROPERTY_WEIGHT:.1%}")
print(f"  Liquid weight:    {LIQUID_WEIGHT_KEEP:.1%}")

Keep — initial wealth: $400,000
  Property weight:  50.0%
  Liquid weight:    50.0%


## Scenario B — Sell and invest

Portfolio:
- 100% liquid diversified portfolio (equities + bonds)
- No property-specific cashflows

Cashflows:
- Annual savings contributions (same as Scenario A)

In [6]:
sell_portfolio = wp.Portfolio(
    slots={
        "US Total Market":     us_total_mkt,
        "US Large Cap Growth": us_lg_growth,
        "NASDAQ 100":          nasdaq,
        "Russell 1000":        russell_1000,
        "Europe Developed":    europe,
        "US Large Cap":        us_large_cap,
    },
    weights=LIQUID_SUB_WEIGHTS,
    name="Sell & Invest",
)

sell_cashflows = [
    wp.cashflows.PeriodicCashflow(
        amount=ANNUAL_SAVINGS,
        frequency="annual",
        mode="dollar",
        real=True,
    ),
]

TOTAL_WEALTH_SELL = SELL_PROCEEDS + LIQUID_WEALTH

sell_sim = wp.analytics.WealthSimulation(
    method=wp.sim.MonteCarlo(seed=42),
    cashflows=sell_cashflows,
    horizon_years=HORIZON_YEARS,
    initial_wealth=TOTAL_WEALTH_SELL,
    n_simulations=N_SIMS,
    inflation_rate=INFLATION_RATE,
)

sell_result = sell_sim.compute(
    sell_portfolio,
    start=HIST_START,
    end=HIST_END,
    frequency="daily",
    start_date=TODAY,
    real=True,
)

print(f"Sell — initial wealth: ${TOTAL_WEALTH_SELL:,.0f}")

Sell — initial wealth: $375,000


## Compare scenarios

In [7]:
comparison = wp.analytics.ComparisonResult.from_scenarios({
    "Keep": keep_result,
    "Sell": sell_result,
})

comparison.summary()

scenario,initial_wealth,p5,p50,p95
str,f64,f64,f64,f64
"""Keep""",400000.0,1.7092e6,4.0986e6,1.0718e7
"""Sell""",375000.0,1.2271e6,4.1612e6,1.4223e7


In [8]:
print(f"P(Keep beats Sell): {comparison.prob_wins('Keep', 'Sell'):.1%}")
print(f"P(Sell beats Keep): {comparison.prob_wins('Sell', 'Keep'):.1%}")

P(Keep beats Sell): 49.4%
P(Sell beats Keep): 50.6%


In [9]:
comparison.plot()

## Individual fan charts

In [10]:
keep_result.plot()

In [11]:
keep_result.plot_allocation()

In [12]:
sell_result.plot()

In [13]:
sell_result.plot_allocation()